In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.master("local[*]").appName("6").getOrCreate()


**Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.**

The Driver controls the Spark application by creating jobs, dividing them into tasks, and coordinating execution. The Cluster Manager allocates resources and launches executors on worker nodes. Executors run the assigned tasks, process data in parallel, store intermediate results when needed, and send the final output back to the Driver.

**Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain
processing large datasets?**

Spark's Lazy Evaluation delays execution until an action is called. Instead of executing each transformation immediately, Spark builds an execution plan (DAG) and optimizes it. This reduces unnecessary computations, minimizes data movement, and combines multiple operations into a single efficient execution, improving performance when processing large datasets.


**Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring 
the first row is treated as a header and inferSchema is enabled.**

In [2]:
# commenting the code for this question as it is only a snippet

# df1 = spark.read.csv("data/source.csv", header=True, inferSchema=True)

**Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. 
columnar) and why does it matter for performance?**

CSV is a row-based storage format that stores data as plain text, making it easy to read but less efficient for large datasets.<br>
Parquet is a columnar storage format that compresses data better and reads only the required columns. This reduces storage space, speeds up queries, and improves Spark's performance for analytical workloads.

**Q5: Given a DataFrame df, write a query to select the columns product_id and price 
where the category is 'Electronics'.**

In [3]:
df= spark.read.csv("ecommerce_dataset.csv",header=True,inferSchema=True)
df.columns

['Product ID',
 'Product Name',
 'Category',
 'Price (USD)',
 'Stock Quantity',
 'Rating',
 'Number of Reviews',
 'Seller Name',
 'Discount Percentage',
 'Sales Count']

In [4]:
df.select("Product ID", "Price (USD)","Category")\
.filter(df["Category"] == "Electronics")\
.show()

+----------+-----------+-----------+
|Product ID|Price (USD)|   Category|
+----------+-----------+-----------+
|     P1007|     155.78|Electronics|
|     P1015|     176.65|Electronics|
|     P1018|     107.41|Electronics|
|     P1021|     176.04|Electronics|
|     P1022|     248.61|Electronics|
|     P1036|     443.76|Electronics|
|     P1067|     341.56|Electronics|
|     P1082|     110.92|Electronics|
|     P1096|      296.0|Electronics|
|     P1098|     447.99|Electronics|
+----------+-----------+-----------+



**Q6: Write the code to "revise" a DataFrame by renaming the column old_name to 
new_name and casting the price column from a String to a Double.**

In [5]:
df_q6= spark.read.csv("products.csv",header=True,inferSchema=False)
df_q6.dtypes

[('old_name', 'string'), ('price', 'string'), ('category', 'string')]

In [6]:
df_q6= df_q6.withColumnRenamed("old_name", "new_name")\
      .withColumn("price", col("price").cast("double"))
df_q6.printSchema()

root
 |-- new_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- category: string (nullable = true)



**Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker 
node fails?**

Spark uses a Lineage Graph (DAG) to track all transformations applied to a dataset. If a worker node fails and data is lost, Spark does not rely on replication. <br>
Instead, it uses the DAG to recompute only the lost partitions from the original data, ensuring fault tolerance while minimizing recomputation and improving efficiency.


**Q8: Write a query to filter a DataFrame df_orders for rows where the status is 
'Completed' AND the amount is greater than 1000.**

In [7]:
df_orders= spark.read.option("multiline", "true").json("orders.json")
df_orders.show()

+------+--------+--------+---------+
|amount|customer|order_id|   status|
+------+--------+--------+---------+
|  1500|   Alice|     101|Completed|
|   850|     Bob|     102|  Pending|
|   950| Charlie|     103|Completed|
|  2500|   David|     104|Completed|
|  1800|     Eva|     105|Cancelled|
|  1200|   Frank|     106|Completed|
+------+--------+--------+---------+



In [8]:
df_orders.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
).show()

+------+--------+--------+---------+
|amount|customer|order_id|   status|
+------+--------+--------+---------+
|  1500|   Alice|     101|Completed|
|  2500|   David|     104|Completed|
|  1200|   Frank|     106|Completed|
+------+--------+--------+---------+



**Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount 
of data loaded into memory.**

Predicate Pushdown is an optimization in Parquet where filter conditions are applied while reading the file. Spark reads only the rows and columns that satisfy the filter instead of scanning the entire dataset. 
1) This reduces disk I/O.<br>
2) loads less data into memory <br>
3) significantly improves query performance.


**Q10: Write a code snippet to add a new column final_price which is the base_price 
multiplied by 1.18 (18% tax).**

In [9]:
#using df_q6(products dataset).
# base_price is price in this dataset

df_q6 = df_q6.withColumn("final_price", col("price") * 1.18)
df_q6.show()

+------------+------+---------------+------------------+
|    new_name| price|       category|       final_price|
+------------+------+---------------+------------------+
|      Laptop|799.99|    Electronics|          943.9882|
|       Shirt|  29.5|       Clothing|34.809999999999995|
|        Book| 15.75|          Books|18.584999999999997|
|  Headphones|149.99|    Electronics|          176.9882|
|       Shoes| 89.95|       Footwear|106.14099999999999|
|Coffee Maker| 59.99|Home Appliances|           70.7882|
|       Watch|199.99|    Accessories|          235.9882|
|    Keyboard|  45.0|    Electronics|53.099999999999994|
|    Backpack| 39.99|           Bags|           47.1882|
|       Mixer| 120.5|Home Appliances|            142.19|
+------------+------+---------------+------------------+



**Q11: What is the difference between Transformations and Actions? Provide two 
examples of each.**

Transformations are operations that create a new DataFrame from an existing one but are not executed immediately (lazy evaluation).<br>
Actions trigger the execution of transformations and produce a result or output.

In [10]:
# Transformation examples:
df_q6.filter(df_q6.price > 100)  # filter all products wit price>100
df_q6.select("new_name", "category") #selects the name and category columns

DataFrame[new_name: string, category: string]

In [11]:
# Action examples
df_q6.show() #displays the DataFrame on the screen
df_q6.count() #total number of rows in the DataFrame

+------------+------+---------------+------------------+
|    new_name| price|       category|       final_price|
+------------+------+---------------+------------------+
|      Laptop|799.99|    Electronics|          943.9882|
|       Shirt|  29.5|       Clothing|34.809999999999995|
|        Book| 15.75|          Books|18.584999999999997|
|  Headphones|149.99|    Electronics|          176.9882|
|       Shoes| 89.95|       Footwear|106.14099999999999|
|Coffee Maker| 59.99|Home Appliances|           70.7882|
|       Watch|199.99|    Accessories|          235.9882|
|    Keyboard|  45.0|    Electronics|53.099999999999994|
|    Backpack| 39.99|           Bags|           47.1882|
|       Mixer| 120.5|Home Appliances|            142.19|
+------------+------+---------------+------------------+



10

**Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out 
any rows where user_id is null, and save the result as a CSV at "path/to/output".**

In [19]:
#commenting the code for this question as it is only a snippet

#df = spark.read.parquet("path/to/input")

#df.filter(col("user_id").isNotNull()) \
# .write.mode("overwrite") \
#  .option("header", True) \
#  .csv("path/to/output")

+-------+-------+---+---------+
|user_id|   name|age|     city|
+-------+-------+---+---------+
|      1|  Alice| 24| New York|
|      2|    Bob| 28|  Chicago|
|   NULL|Charlie| 22|   Boston|
|      4|  David| 31|  Seattle|
|   NULL|    Eva| 26|    Miami|
|      6|  Frank| 35|  Houston|
|      7|  Grace| 29|   Denver|
|   NULL|  Henry| 27|   Austin|
|      9|    Ivy| 23|San Diego|
|     10|   Jack| 40|   Dallas|
|     11|   Kate| 32|  Phoenix|
|   NULL|   Liam| 21|  Atlanta|
|     13|    Mia| 25|  Orlando|
|     14|   Noah| 36|  Detroit|
|   NULL| Olivia| 30| San Jose|
|     16|  Peter| 34|Las Vegas|
|     17|  Quinn| 27| Portland|
|     18| Rachel| 33|Nashville|
|   NULL|    Sam| 24|    Tampa|
|     20|   Tina| 29|Charlotte|
+-------+-------+---+---------+



**Q13: In Spark Architecture, what is the difference between Client Mode and Cluster 
Mode?**

In Client Mode, the Driver runs on the user's local machine and communicates with the cluster to execute tasks.<br> In Cluster Mode, the Driver runs inside the cluster on a worker node, making the application more reliable and suitable for production since it continues running even if the client disconnects.

**Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority 
is 'High'.**

In [17]:
# the dataset used (support tickets) does not have north region
df_support= spark.read.csv("Support_tickets.csv", header=True,inferSchema= True)
df_support.select("region").distinct().show()
df_support.select("priority").distinct().show()

+------+
|region|
+------+
|  AMER|
|  APAC|
|  EMEA|
+------+

+--------+
|priority|
+--------+
|     low|
|    high|
|  medium|
+--------+



In [18]:
df_support.filter((col("region") == 'AMER') | (col("priority")=='high'))\
          .select("ticket_id", "region", "priority").show()

+----------+------+--------+
| ticket_id|region|priority|
+----------+------+--------+
|1000000001|  AMER|     low|
|1000000003|  AMER|  medium|
|1000000004|  AMER|     low|
|1000000008|  AMER|  medium|
|1000000011|  AMER|  medium|
|1000000019|  AMER|    high|
|1000000022|  AMER|     low|
|1000000023|  AMER|     low|
|1000000025|  AMER|  medium|
|1000000027|  AMER|     low|
|1000000028|  AMER|     low|
|1000000030|  AMER|  medium|
|1000000031|  EMEA|    high|
|1000000035|  AMER|    high|
|1000000036|  AMER|     low|
|1000000038|  AMER|     low|
|1000000039|  AMER|  medium|
|1000000040|  AMER|    high|
|1000000043|  AMER|  medium|
|1000000044|  AMER|    high|
+----------+------+--------+
only showing top 20 rows


**Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on 
a multi-terabyte dataset?**

Using `.show(5)` is safer because it retrieves and displays only the first five rows of the dataset. In contrast, `.collect()` loads all the data into the Driver's memory. On a multi-terabyte dataset, this can cause excessive memory usage, slow performance, or even crash the application due to an OutOfMemory error.
